# 2025.1.22 Experiments

We would like to show three things in this jupyter notebook. 

1. Steady flow with drag 
2. Add additional friction constant
3. Add another slip/displacement force. 

First lets import the relevant files and choose the proper directory. 

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Modify base path for depending on your file structure.
BASE_PATH = "/Users/paxton/git"

# Specify path where .pkl files are located
target_dir = f"{BASE_PATH}/quail_volcano/scenarios/simple_1D_test"
# Specify path for Quail source code
source_dir = f"{BASE_PATH}/quail_volcano/src"
# Change to working directory
os.chdir(target_dir)


# Import quail modules
os.chdir(source_dir)

import meshing.tools as mesh_tools

import numerics.helpers.helpers as helpers
import numerics.timestepping.tools as stepper_tools

import physics.zerodimensional.zerodimensional as zerod
import physics.euler.euler as euler
import physics.navierstokes.navierstokes as navierstokes
import physics.scalar.scalar as scalar
import physics.chemistry.chemistry as chemistry
import physics.multiphasevpT.multiphasevpT as multiphasevpT

import processing.readwritedatafiles as readwritedatafiles
import processing.post as post
import processing.plot as plot

import solver.DG as DG
import solver.ADERDG as ADERDG
import solver.tools as solver_tools

import time
import multiprocessing as mp  
from multidomain import Domain, Observer

os.chdir(target_dir)

In [34]:
from IPython.display import HTML
from matplotlib import animation

def animate_conduit_pressure(folder, iterations=100):
    fig = plt.figure(figsize=(8,6.4))
    ax = fig.add_subplot(211,autoscale_on=False,\
                            xlim=(0,-1000),ylim=(0,10))
    
    ax2 = fig.add_subplot(212,autoscale_on=False,\
                            xlim=(0,-1000),ylim=(-1,1))

    pressure_line,  = ax.plot([], [], color="blue", label="pressure")
    velocity_line, = ax2.plot([], [], color="red", label="velocity")

    ax.set_xlabel("Depth (m)")
    ax.set_ylabel("Pressure (MPa)")
    ax2.set_ylabel("Velocity (m/s)")

    time_template = 'time = %.2f [s]'
    time_text = ax.text(0.7,0.9,'',transform=ax.transAxes)

    def init():
        pressure_line.set_data([], [])
        velocity_line.set_data([], [])
        time_text.set_text("")
        return pressure_line, velocity_line, time_text

    def animate(i):
        solver = readwritedatafiles.read_data_file(f"{folder}/test_output_{i}.pkl")
        flag_non_physical = True
        p = solver.physics.compute_additional_variable("Pressure", solver.state_coeffs, flag_non_physical)
        v = solver.physics.compute_additional_variable("Velocity", solver.state_coeffs, flag_non_physical)

        # Get the position of of each nodal points (location corresponding to each entry of pDensityX)
        nodal_pts = solver.basis.get_nodes(solver.order)
        # Allocate [ne] x [nb, ndims]
        x = np.empty((solver.mesh.num_elems,) + nodal_pts.shape)
        for elem_ID in range(solver.mesh.num_elems):
            # Fill coordinates in physical space
            x[elem_ID] = mesh_tools.ref_to_phys(solver.mesh, elem_ID, nodal_pts)
        
        pressure_line.set_data(x.ravel(), p.ravel()/1e6)
        velocity_line.set_data(x.ravel(), v.ravel())
        time_text.set_text(time_template % solver.time)

        return pressure_line, velocity_line, time_text

    plt.close()
    return animation.FuncAnimation(fig, animate, np.arange(iterations), blit=False, init_func=init, interval=50)

## 1. Steady state flow

- No water, no air, no crystals 
- constant viscosity. 
- g = 0 

### Conservation of momentum 
$$
\rho (\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial y}) = - \frac{\partial p}{\partial y} - \rho g - \frac{2 \tau}{R}
$$

The term on the left is the inertial term, and the terms on the right are pressure gradient, weight, and drag. For the sake of our rough analytical solution, we assume that the fluid is largely incompressible and as a result, we expect the steady state of have a constant velocity with respect to both time and space. 

$$
0 = - \frac{\partial p}{\partial x} - \frac{2 \tau}{R}
$$

For laminar flows, $tau = \frac{4 \mu u}{R}$ where $\mu$ is viscosity. For our steady state flow, pressure changes with respect to x linearly. $\frac{\partial p}{\partial t} = \frac{P_L - P_0}{L}$. Plugging those quantities into the above equation, we get. 

$$
\begin{align}
0 &= \frac{P_L - P_0}{L} - \frac{8\mu u}{R^2} \\

u &= \frac{P_L - P_0}{L} \frac{R^2}{8 \mu}
\end{align}
$$

It is worth noting that because the fluid is compressible, in reality $p = p(\rho)$ and specifically pressure increases slightly with density. However, our analytical solution above should still be approximately correct given that our fluid is not that compressible. 

A plausible estimate for $\mu = 1e17 [Pa * s]$, $R = 50m$, and $P_L = 3e6$, and $P_0 = $

In [35]:
# Still in the works 
HTML(animate_conduit_pressure("steady_state_flow").to_html5_video())